# Lesson 1 — Linear Algebra for Machine Learning

**Why this matters:** In ML, *everything is a vector or a matrix*.
- One customer / one image / one sentence → a **vector** of numbers
- A whole dataset → a **matrix** (rows = samples, columns = features)
- A model's prediction → a **matrix × vector** multiplication
- Dimensionality reduction (PCA) → **eigenvectors**

Setup (run once in a terminal): `pip install numpy matplotlib`

Work through each section: read → run → change the numbers → do the exercise.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. Vectors — one data point

A customer with `age=25`, `income=40 (k$)`, `purchases=3` is the vector
$\mathbf{x} = [25, 40, 3]$. Each number is one **feature**, and the vector lives in $\mathbb{R}^3$.

In [ ]:
a = np.array([25, 40, 3])
b = np.array([30, 55, 5])
c = a.reshape(3,1)
print("a + b       =", a + b)       # element-wise addition
print("2 * a       =", 2 * a)       # scalar multiplication
print("b - a       =", b - a)       # difference between two customers
print("shape of a  =", a.shape)
print("re_shape of c  =", c)
print("New shape:", c.shape) 

a + b       = [55 95  8]
2 * a       = [50 80  6]
b - a       = [ 5 15  2]
shape of a  = (3,)
re_shape of c  = [[25 40  3]]
New shape: (1, 3)


### Norm (length) and distance

$$\|\mathbf{x}\| = \sqrt{x_1^2 + x_2^2 + \dots + x_n^2}$$

The distance between two points is $\|\mathbf{a}-\mathbf{b}\|$. This is exactly what **k-Nearest Neighbors** and **k-Means** use.

In [ ]:
length_a = np.sqrt(np.sum(a**2))          # by hand
print("||a|| by hand   :", length_a)
print("||a|| numpy     :", np.linalg.norm(a))
print("distance(a, b)  :", np.linalg.norm(a - b))

### Dot product and cosine similarity

$$\mathbf{a}\cdot\mathbf{b} = \sum_i a_i b_i = \|\mathbf{a}\|\|\mathbf{b}\|\cos\theta$$

- Large positive → vectors point the same way (similar)
- 0 → perpendicular (unrelated)
- Negative → opposite

**Cosine similarity** $= \dfrac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$ is used in recommender systems, search and text embeddings.

In [ ]:
def cosine_similarity(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

# Movie ratings by 3 users on 4 movies (action, action, romance, romance)
alice = np.array([5, 4, 1, 0])
bob   = np.array([4, 5, 0, 1])
carol = np.array([0, 1, 5, 4])

print("a·b by hand:", sum(x * y for x, y in zip(alice, bob)))
print("a·b numpy  :", alice @ bob)          # @ is the dot / matrix-multiply operator
print("sim(alice, bob)   =", round(cosine_similarity(alice, bob), 3))
print("sim(alice, carol) =", round(cosine_similarity(alice, carol), 3))

**Exercise 1.** Add a user `dave = np.array([3, 3, 3, 3])`. Who is Dave most similar to? Predict first, then compute.

In [ ]:
# Your code here

## 2. Matrices — the whole dataset

A dataset with $n$ samples and $d$ features is a matrix $X \in \mathbb{R}^{n \times d}$.
Row $i$ = sample $i$; column $j$ = feature $j$.

In [ ]:
# 4 houses: [size (100 m²), bedrooms, age (10 years)]
X = np.array([
    [1.0, 2, 1.5],
    [1.5, 3, 0.5],
    [2.0, 3, 2.0],
    [2.5, 4, 1.0],
])
print("X shape (n_samples, n_features):", X.shape)
print("first sample (row 0):   ", X[0])
print("feature 'size' (col 0): ", X[:, 0])
print("X transposed shape:", X.T.shape)

## 3. Matrix × vector = predictions of a linear model

A linear model predicts $\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + b$, which is one dot product per sample.
For **all samples at once**:

$$\hat{\mathbf{y}} = X\mathbf{w} + b$$

Shapes: $(n \times d)(d) \rightarrow (n)$. **Always check shapes**; most ML bugs are shape bugs.

In [ ]:
w = np.array([100, 20, -15])   # price per unit of each feature (k$)
b = 50                         # base price

# Loop version (slow, but shows the idea)
y_loop = np.array([np.dot(row, w) + b for row in X])

# Vectorized version (how ML libraries do it)
y_hat = X @ w + b

print("loop      :", y_loop)
print("vectorized:", y_hat)

### Matrix × matrix

$(A B)_{ij}$ = dot product of **row $i$ of $A$** with **column $j$ of $B$**. It requires `A.shape[1] == B.shape[0]`.
A neural-network layer is exactly this: `output = X @ W + b`.

In [ ]:
W = np.random.default_rng(0).normal(size=(3, 5))   # a layer: 3 inputs -> 5 neurons
H = X @ W
print(f"{X.shape} @ {W.shape} -> {H.shape}")

A = np.array([[1, 2], [3, 4]])
B = np.array([[0, 1], [1, 0]])
print("A @ B =\n", A @ B)
print("B @ A =\n", B @ A, "\n-> matrix multiplication is NOT commutative")

**Exercise 2.** Without running code, what is the shape of `X.T @ X`? Of `X @ X.T`? Then check with code.

In [ ]:
# Your code here

## 4. Inverse & solving systems → Linear Regression from scratch

The identity $I$ satisfies $AI = A$. The inverse satisfies $A^{-1}A = I$ (only for square, non-singular matrices).

Linear regression finds weights that minimize squared error. The closed-form solution (**normal equation**) is:

$$\mathbf{w} = (X^\top X)^{-1} X^\top \mathbf{y}$$

In practice use `np.linalg.solve` or `np.linalg.lstsq`; they are more numerically stable than computing an inverse.

In [ ]:
rng = np.random.default_rng(42)
n = 100
size = rng.uniform(0.5, 3.0, n)                    # house size (100 m²)
price = 80 * size + 30 + rng.normal(0, 10, n)      # true: slope 80, intercept 30, plus noise

# Add a column of 1s so the intercept becomes just another weight
X_design = np.column_stack([np.ones(n), size])     # shape (100, 2)

w_normal = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ price
w_solve  = np.linalg.solve(X_design.T @ X_design, X_design.T @ price)
w_lstsq, *_ = np.linalg.lstsq(X_design, price, rcond=None)

print("normal equation [intercept, slope]:", w_normal)
print("np.linalg.solve                   :", w_solve)
print("np.linalg.lstsq                   :", w_lstsq)

In [ ]:
xs = np.linspace(0.5, 3.0, 50)
plt.scatter(size, price, alpha=0.6, label="data")
plt.plot(xs, w_lstsq[0] + w_lstsq[1] * xs, color="red", label="fitted line")
plt.xlabel("size (100 m²)"); plt.ylabel("price (k$)"); plt.legend()
plt.title("Linear regression = linear algebra")
plt.show()

**Exercise 3.** Add a second feature `bedrooms = rng.integers(1, 5, n)` and make the true price depend on it too
(e.g. `+ 15 * bedrooms`). Rebuild `X_design` with 3 columns and check that you recover all 3 weights.

In [ ]:
# Your code here

## 5. Eigenvalues & eigenvectors → PCA from scratch

An eigenvector $\mathbf{v}$ of matrix $A$ only gets *stretched* (not rotated) by $A$:

$$A\mathbf{v} = \lambda \mathbf{v}$$

$\lambda$ is the eigenvalue (the stretch factor).

**PCA idea:** the eigenvectors of the data's **covariance matrix** are the directions of greatest spread, and the eigenvalues tell how much variance lies along each direction.
Projecting onto the top-$k$ eigenvectors reduces dimensions while keeping the most information.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
eigvals, eigvecs = np.linalg.eigh(A)   # eigh: for symmetric matrices, returns real values (ascending)
print("eigenvalues:", eigvals)
print("eigenvectors (columns):\n", eigvecs)

v = eigvecs[:, 0]
print("A @ v      =", A @ v)
print("lambda * v =", eigvals[0] * v, " <- same!")

In [ ]:
# Correlated 2-D data (e.g. height & weight)
rng = np.random.default_rng(1)
data = rng.multivariate_normal(mean=[0, 0], cov=[[3, 2.5], [2.5, 3]], size=300)

# PCA step by step
centered = data - data.mean(axis=0)                  # 1. center
cov = centered.T @ centered / (len(data) - 1)        # 2. covariance matrix (same as np.cov(centered.T))
eigvals, eigvecs = np.linalg.eigh(cov)               # 3. eigen-decomposition (eigh: for symmetric matrices)
order = np.argsort(eigvals)[::-1]                    # 4. sort by variance, largest first
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

explained = eigvals / eigvals.sum()
print("explained variance ratio:", explained)

projected_1d = centered @ eigvecs[:, :1]             # 5. project 2-D -> 1-D
print("original shape:", data.shape, "-> reduced shape:", projected_1d.shape)

plt.scatter(centered[:, 0], centered[:, 1], alpha=0.3)
for val, vec in zip(eigvals, eigvecs.T):
    plt.arrow(0, 0, *(vec * 2 * np.sqrt(val)), width=0.05, color="red")
plt.axis("equal"); plt.title("Principal components = eigenvectors of covariance")
plt.show()

**Exercise 4.** Change `cov` to `[[3, 0], [0, 3]]` (no correlation). What happens to the explained variance ratio? Why does PCA not help here?

## Summary

| Concept | ML use |
|---|---|
| Vector, norm, distance | features, k-NN, k-Means |
| Dot product, cosine similarity | linear models, recommenders, embeddings |
| Matrix × vector / matrix | predictions, neural-network layers |
| Inverse, solving systems | linear regression (normal equation) |
| Eigenvectors | PCA, dimensionality reduction |

**Next lesson:** Calculus → derivatives, gradients and **gradient descent** (how models actually learn).

In [ ]:
# Your code here